In [7]:

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

IEEE_SINGLE_COLUMN_WIDTH = 3.5
IEEE_DOUBLE_COLUMN_WIDTH = 7.16

PAPER_RC = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size': 8.5,
    'axes.labelsize': 8,
    'axes.titlesize': 8.5,
    'axes.linewidth': 0.6,
    'legend.fontsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
}

PLOT_STYLE = {
    'gt_color': '#111111',
    'candidate_color_start': '#d8a600',
    'candidate_color_end': '#5f6670',
    'candidate_alpha_min': 0.18,
    'candidate_alpha_max': 0.42,
    'best_color': '#1f4e79',
    'joint_edge': '#f8f8f8',
    'axis_edge': '0.82',
    'label_color': '0.18',
}

CANDIDATE_CMAP = LinearSegmentedColormap.from_list(
    'candidate_yellow_to_grey',
    [PLOT_STYLE['candidate_color_start'], PLOT_STYLE['candidate_color_end']],
)

plt.rcParams.update(PAPER_RC)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'results').exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.dataset_graphs import get_root_first_single_hand_graph, resolve_local_hand_graph_metadata

EVAL_SAMPLES_ROOT = PROJECT_ROOT / 'out' / 'diffusion_hands_runs'


def _display_model_name(model_name):
    model_name = str(model_name or '')
    if model_name in {'twostage_dct_diffusion', 'card'}:
        return 'STAR-hand'
    return model_name[:1].upper() + model_name[1:] if model_name else model_name


def _normalize_model_filter(model_filter):
    if model_filter is None or model_filter == 'all':
        return None

    if isinstance(model_filter, str):
        selected_models = [model_filter]
    else:
        selected_models = [str(model_name) for model_name in model_filter]

    normalized = {model_name.strip().lower() for model_name in selected_models if str(model_name).strip()}
    if normalized & {'card', 'twostage_dct_diffusion'}:
        normalized.update({'card', 'twostage_dct_diffusion'})
    return normalized


def _matches_model_filter(model_name, model_filter):
    normalized = _normalize_model_filter(model_filter)
    if normalized is None:
        return True
    return (
        str(model_name or '').strip().lower() in normalized
        or _display_model_name(model_name).lower() in normalized
    )


def list_saved_eval_sample_files(root=EVAL_SAMPLES_ROOT):
    return sorted(root.glob('**/eval_samples.npz'))


def load_saved_eval_samples(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    payload = {key: data[key] for key in data.files if key != 'metadata_json'}
    metadata = {}
    if 'metadata_json' in data.files:
        metadata = json.loads(str(data['metadata_json']))
    pred = payload.get('pred')
    pred_all = payload.get('pred_all')
    if pred is not None and pred_all is not None and pred_all.ndim == 5 and pred.ndim == 4:
        if pred_all.shape[0] != pred.shape[0] and pred_all.shape[1] == pred.shape[0]:
            payload['pred_all'] = np.transpose(pred_all, (1, 0, 2, 3, 4))
    return payload, metadata


def collect_saved_eval_samples(root=EVAL_SAMPLES_ROOT, dataset=None, action_filter=None):
    bundles = []
    for npz_path in list_saved_eval_sample_files(root):
        payload, metadata = load_saved_eval_samples(npz_path)
        meta_dataset = metadata.get('dataset')
        meta_action = metadata.get('action_filter', '') or 'all'
        if dataset is not None and meta_dataset != dataset:
            continue
        if action_filter is not None and meta_action != action_filter:
            continue
        bundles.append({'path': npz_path, 'payload': payload, 'metadata': metadata})
    return bundles


def _select_latest_bundle_per_model(bundles):
    by_model = {}
    for bundle in bundles:
        model_name = bundle['metadata'].get('model', bundle['path'].parent.name)
        current = by_model.get(model_name)
        if current is None or bundle['path'].stat().st_mtime > current['path'].stat().st_mtime:
            by_model[model_name] = bundle
    return dict(sorted(by_model.items()))


def _sample_timestep_indices(num_frames, max_timesteps=10):
    max_timesteps = max(1, min(int(max_timesteps), int(num_frames)))
    if max_timesteps == num_frames:
        return np.arange(num_frames, dtype=int)
    return np.unique(np.round(np.linspace(0, num_frames - 1, max_timesteps)).astype(int))


def _hand_edges_for_dataset(dataset_name):
    if dataset_name is None:
        dataset_name = 'assembly'
    metadata = resolve_local_hand_graph_metadata(str(dataset_name).lower())
    return [tuple(edge) for edge in metadata.get('links', ())]


def _root_first_hand_edges_for_dataset(dataset_name):
    if dataset_name is None:
        dataset_name = 'assembly'
    metadata = get_root_first_single_hand_graph(str(dataset_name).lower())
    return [tuple(edge) for edge in metadata.get('links', ())]


def _model_uses_root_first_graph(model_name):
    model_name = str(model_name or '').lower()
    return model_name in {'dlow_cvae', 'humanmac'}


def _hand_edges_for_plot(model_name, dataset_name):
    use_root_first = _model_uses_root_first_graph(model_name)
    return _root_first_hand_edges_for_dataset(dataset_name) if use_root_first else _hand_edges_for_dataset(dataset_name)


def _prepare_hands_for_plot(hands_frame, model_name, dataset_name, hand_edges=None):
    hands_frame = np.asarray(hands_frame)
    if hands_frame.ndim == 2:
        hands_frame = hands_frame[None, ...]

    use_root_first = _model_uses_root_first_graph(model_name)
    if hand_edges is None:
        hand_edges = _hand_edges_for_plot(model_name, dataset_name)

    if use_root_first and hands_frame.shape[-2] == 20:
        root = np.zeros(hands_frame.shape[:-2] + (1, hands_frame.shape[-1]), dtype=hands_frame.dtype)
        hands_frame = np.concatenate([root, hands_frame], axis=-2)

    return hands_frame, hand_edges


def _compute_axis_box(points):
    points = np.asarray(points)
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = 0.5 * (mins + maxs)
    radius = max(1e-6, 0.5 * np.max(maxs - mins))
    return center, radius


def _set_equal_axes(ax, points=None, axis_box=None):
    if axis_box is None:
        if points is None:
            raise ValueError('Either points or axis_box must be provided.')
        axis_box = _compute_axis_box(points)
    center, radius = axis_box
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def _style_hand_axis(ax, style=PLOT_STYLE):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)
    ax.set_facecolor('white')
    ax.set_box_aspect((1, 1, 1))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_alpha(0.0)
        axis.pane.set_edgecolor(style['axis_edge'])
        if hasattr(axis, 'line'):
            axis.line.set_color(style['axis_edge'])
            axis.line.set_linewidth(0.35)


def _add_row_label(ax, label, style=PLOT_STYLE):
    ax.text2D(
        -0.10,
        0.5,
        label,
        transform=ax.transAxes,
        rotation=90,
        va='center',
        ha='center',
        color=style['label_color'],
        fontsize=8.5,
        fontweight='semibold',
    )


def _downsample_hands(hands_frame, max_hands_per_subplot=None):
    hands_frame = np.asarray(hands_frame)
    if hands_frame.ndim == 2:
        return hands_frame[None, ...]
    if max_hands_per_subplot is None:
        return hands_frame
    max_hands_per_subplot = max(1, int(max_hands_per_subplot))
    if hands_frame.shape[0] <= max_hands_per_subplot:
        return hands_frame
    keep_idx = np.linspace(0, hands_frame.shape[0] - 1, max_hands_per_subplot).round().astype(int)
    keep_idx = np.unique(keep_idx)
    return hands_frame[keep_idx]


def _find_best_candidate_index(pred_all_sample, target_sample, model_name=None, dataset_name=None):
    pred_all_sample, _ = _prepare_hands_for_plot(pred_all_sample, model_name=model_name, dataset_name=dataset_name)
    target_sample, _ = _prepare_hands_for_plot(target_sample, model_name=model_name, dataset_name=dataset_name)
    pred_all_sample = np.asarray(pred_all_sample)
    target_sample = np.asarray(target_sample)
    if pred_all_sample.ndim < 4:
        return None
    diffs = pred_all_sample - target_sample[None, ...]
    distances = np.linalg.norm(diffs, axis=-1).mean(axis=tuple(range(1, diffs.ndim - 1)))
    return int(np.argmin(distances))


def _plot_hand_frame(ax, hands_frame, hand_edges, title=None, hand_cmap=None, max_hands_per_subplot=32, fixed_color=None, best_idx=None, axis_box=None, style=PLOT_STYLE):
    hands_frame = np.asarray(hands_frame)
    hands_frame = _downsample_hands(hands_frame, max_hands_per_subplot=max_hands_per_subplot)

    if fixed_color is not None:
        hand_styles = [{'color': fixed_color, 'alpha': 1.0, 'linewidth': 0.95, 'size': 10, 'zorder': 3}] * hands_frame.shape[0]
    elif hands_frame.shape[0] == 1:
        hand_styles = [{'color': style['best_color'], 'alpha': 1.0, 'linewidth': 0.95, 'size': 10, 'zorder': 3}]
    else:
        if hand_cmap is None:
            hand_cmap = CANDIDATE_CMAP
        hand_colors = hand_cmap(np.linspace(0.0, 1.0, hands_frame.shape[0]))
        hand_alphas = np.linspace(style['candidate_alpha_min'], style['candidate_alpha_max'], hands_frame.shape[0])
        hand_styles = [
            {'color': hand_colors[idx], 'alpha': hand_alphas[idx], 'linewidth': 0.45, 'size': 5, 'zorder': 1}
            for idx in range(hands_frame.shape[0])
        ]
        if best_idx is not None and 0 <= int(best_idx) < hands_frame.shape[0]:
            hand_styles[int(best_idx)] = {'color': style['best_color'], 'alpha': 1.0, 'linewidth': 1.05, 'size': 11, 'zorder': 4}

    draw_order = [idx for idx in range(hands_frame.shape[0]) if idx != best_idx]
    if best_idx is not None and 0 <= int(best_idx) < hands_frame.shape[0]:
        draw_order.append(int(best_idx))

    for hand_idx in draw_order:
        hand = hands_frame[hand_idx]
        hand_style = hand_styles[hand_idx]
        valid_edges = [(i, j) for i, j in hand_edges if i < hand.shape[0] and j < hand.shape[0]]
        for i, j in valid_edges:
            ax.plot(
                [hand[i, 0], hand[j, 0]],
                [hand[i, 1], hand[j, 1]],
                [hand[i, 2], hand[j, 2]],
                color=hand_style['color'],
                alpha=hand_style['alpha'],
                linewidth=hand_style['linewidth'],
                zorder=hand_style['zorder'],
            )
        ax.scatter(
            hand[:, 0],
            hand[:, 1],
            hand[:, 2],
            color=hand_style['color'],
            alpha=hand_style['alpha'],
            s=hand_style['size'],
            edgecolors=style['joint_edge'],
            linewidths=0.18,
            depthshade=False,
            zorder=hand_style['zorder'],
        )

    ax.scatter([0], [0], [0], color=style['gt_color'], s=8, depthshade=False, zorder=5)

    all_points = hands_frame.reshape(-1, 3)
    _set_equal_axes(ax, all_points, axis_box=axis_box)
    ax.view_init(elev=20, azim=-65)
    _style_hand_axis(ax, style=style)
    if title:
        ax.set_title(title, pad=1.5, color=style['label_color'], fontweight='semibold')


def visualize_saved_eval_grid(sample_idx=0, dataset=None, action_filter=None, max_timesteps=10, model_order=None, model_filter='all', max_hands_per_subplot=32):
    bundles = collect_saved_eval_samples(dataset=dataset, action_filter=action_filter)
    bundles_by_model = _select_latest_bundle_per_model(bundles)
    if not bundles_by_model:
        raise ValueError('No saved eval samples found for the requested filters.')

    if model_order is None:
        candidate_model_names = list(bundles_by_model.keys())
    else:
        candidate_model_names = [name for name in model_order if name in bundles_by_model]

    model_names = [name for name in candidate_model_names if _matches_model_filter(name, model_filter)]
    if not model_names:
        available_models = ', '.join(_display_model_name(name) for name in candidate_model_names)
        raise ValueError(f'No models matched model_filter={model_filter!r}. Available models: {available_models}')

    first_bundle = bundles_by_model[model_names[0]]
    first_payload = first_bundle['payload']
    if sample_idx >= len(first_payload["target"]):
        raise IndexError(
            f"sample_idx={sample_idx} out of range for {len(first_payload['target'])} saved samples."
        )

    dataset_name = first_bundle['metadata'].get('dataset', dataset or 'assembly')
    hand_edges_gt = _hand_edges_for_plot('ground_truth', dataset_name)

    num_future_frames = first_payload['target'].shape[1]
    timestep_indices = _sample_timestep_indices(num_future_frames, max_timesteps=max_timesteps)
    print(f'Sampled timesteps: {timestep_indices.tolist()}')
    num_rows = 1 + len(model_names)
    num_cols = len(timestep_indices)
    fig_width = IEEE_DOUBLE_COLUMN_WIDTH
    subplot_size = fig_width / max(num_cols + 0.35, 1)
    fig_height = max(IEEE_SINGLE_COLUMN_WIDTH, subplot_size * num_rows * 0.92)
    fig, axes = plt.subplots(
        num_rows,
        num_cols,
        figsize=(fig_width, fig_height),
        subplot_kw={'projection': '3d'},
        squeeze=False,
    )
    fig.patch.set_facecolor('white')

    gt_hands = first_payload['target'][sample_idx]
    gt_axis_boxes = {}
    for col_idx, frame_idx in enumerate(timestep_indices):
        ax = axes[0, col_idx]
        gt_frame, gt_edges = _prepare_hands_for_plot(gt_hands[frame_idx], model_name='ground_truth', dataset_name=dataset_name, hand_edges=hand_edges_gt)
        gt_axis_boxes[int(frame_idx)] = _compute_axis_box(gt_frame.reshape(-1, 3))
        gt_title = f't={frame_idx}'
        _plot_hand_frame(ax, gt_frame, gt_edges, title=gt_title, max_hands_per_subplot=max_hands_per_subplot, fixed_color=PLOT_STYLE['gt_color'], axis_box=gt_axis_boxes[int(frame_idx)])

    _add_row_label(axes[0, 0], 'Ground Truth')

    for row_idx, model_name in enumerate(model_names, start=1):
        payload = bundles_by_model[model_name]['payload']
        model_dataset_name = bundles_by_model[model_name]['metadata'].get('dataset', dataset_name)
        hand_edges_model = _hand_edges_for_plot(model_name, model_dataset_name)
        pred_all = payload.get('pred_all')
        pred = payload['pred']
        best_idx = None
        if pred_all is not None:
            best_idx = _find_best_candidate_index(pred_all[sample_idx], payload['target'][sample_idx], model_name=model_name, dataset_name=model_dataset_name)
        for col_idx, frame_idx in enumerate(timestep_indices):
            ax = axes[row_idx, col_idx]
            if pred_all is not None:
                hands_frame = pred_all[sample_idx, :, frame_idx]
            else:
                hands_frame = pred[sample_idx, frame_idx][None, ...]
            hands_frame, hand_edges_model = _prepare_hands_for_plot(hands_frame, model_name=model_name, dataset_name=model_dataset_name, hand_edges=hand_edges_model)
            _plot_hand_frame(ax, hands_frame, hand_edges_model, max_hands_per_subplot=max_hands_per_subplot, best_idx=best_idx, axis_box=gt_axis_boxes[int(frame_idx)])

        _add_row_label(axes[row_idx, 0], _display_model_name(model_name))

    action_name = first_bundle['metadata'].get('action_filter', action_filter or '') or 'all'
    # fig.subplots_adjust(left=0.08, right=0.995, top=0.94, bottom=0.03, wspace=-0.02, hspace=-0.04)
    plt.show()


saved_eval_files = list_saved_eval_sample_files()
print(f'Found {len(saved_eval_files)} eval sample bundles under {EVAL_SAMPLES_ROOT}')

model_filter =  ['all'] # e.g. 'all', 'CARD', ['gsps', 'humanmac']

# Example:
visualize_saved_eval_grid(
    sample_idx=0,
    dataset='assembly',
    action_filter='pick_up_screwd',
    max_timesteps=6,
    model_filter=model_filter,
)


Found 68 eval sample bundles under /home/fagnelli/diffusion_hands/out/diffusion_hands_runs


ValueError: No models matched model_filter=['all']. Available models: Belfusion, Comusion, Gsps, Humanmac, STAR-hand

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
import math


VIDEO_STYLE = {
    **PLOT_STYLE,
    "gt_color": "#111111",
    "best_color": "#1f77b4",
    "shadow_color": "#707070",
    "shadow_alpha": 0.10,
    "shadow_linewidth": 0.45,
    "shadow_joint_size": 4,
    "best_linewidth": 1.35,
    "best_joint_size": 13,
    "gt_linewidth": 1.35,
    "gt_joint_size": 13,
}


def _compute_sequence_axis_box(sequences, padding=0.08):
    """
    Compute one fixed cubic axis box from one or more sequences.

    Each item may have shape:
        [T, J, 3]
        [K, T, J, 3]
    """
    valid_points = []

    for sequence in sequences:
        if sequence is None:
            continue

        sequence = np.asarray(sequence)
        if sequence.size == 0:
            continue

        points = sequence.reshape(-1, 3)
        points = points[np.isfinite(points).all(axis=1)]

        if len(points) > 0:
            valid_points.append(points)

    if not valid_points:
        raise ValueError("No finite points were available to compute the axis limits.")

    points = np.concatenate(valid_points, axis=0)
    center, radius = _compute_axis_box(points)
    radius *= 1.0 + float(padding)

    return center, radius


def _draw_single_hand(
    ax,
    hand,
    hand_edges,
    color,
    alpha,
    linewidth,
    joint_size,
    zorder,
    style=VIDEO_STYLE,
):
    hand = np.asarray(hand)

    valid_edges = [
        (i, j)
        for i, j in hand_edges
        if i < hand.shape[0] and j < hand.shape[0]
    ]

    artists = []

    for i, j in valid_edges:
        line, = ax.plot(
            [hand[i, 0], hand[j, 0]],
            [hand[i, 1], hand[j, 1]],
            [hand[i, 2], hand[j, 2]],
            color=color,
            alpha=alpha,
            linewidth=linewidth,
            zorder=zorder,
        )
        artists.append(line)

    scatter = ax.scatter(
        hand[:, 0],
        hand[:, 1],
        hand[:, 2],
        color=color,
        alpha=alpha,
        s=joint_size,
        edgecolors=style["joint_edge"],
        linewidths=0.18,
        depthshade=False,
        zorder=zorder,
    )
    artists.append(scatter)

    return artists


def _draw_video_frame(
    ax,
    hands_frame,
    hand_edges,
    axis_box,
    title,
    best_idx=None,
    ground_truth=False,
    max_shadow_hands=32,
    elev=20,
    azim=-65,
    style=VIDEO_STYLE,
):
    """
    Draw one video frame.

    Parameters
    ----------
    hands_frame:
        [J, 3] for GT/deterministic output, or [K, J, 3] for stochastic output.
    best_idx:
        Index of the sample closest to GT, computed over the full future sequence.
    """
    ax.cla()

    hands_frame = np.asarray(hands_frame)
    if hands_frame.ndim == 2:
        hands_frame = hands_frame[None, ...]

    if ground_truth:
        _draw_single_hand(
            ax=ax,
            hand=hands_frame[0],
            hand_edges=hand_edges,
            color=style["gt_color"],
            alpha=1.0,
            linewidth=style["gt_linewidth"],
            joint_size=style["gt_joint_size"],
            zorder=5,
            style=style,
        )

    elif hands_frame.shape[0] == 1:
        # Deterministic competitor: its only prediction is also its closest sample.
        _draw_single_hand(
            ax=ax,
            hand=hands_frame[0],
            hand_edges=hand_edges,
            color=style["best_color"],
            alpha=1.0,
            linewidth=style["best_linewidth"],
            joint_size=style["best_joint_size"],
            zorder=5,
            style=style,
        )

    else:
        best_idx = int(best_idx) if best_idx is not None else 0

        shadow_indices = np.asarray(
            [idx for idx in range(hands_frame.shape[0]) if idx != best_idx],
            dtype=int,
        )

        if max_shadow_hands is not None:
            max_shadow_hands = max(0, int(max_shadow_hands))

            if len(shadow_indices) > max_shadow_hands:
                keep = np.linspace(
                    0,
                    len(shadow_indices) - 1,
                    max_shadow_hands,
                ).round().astype(int)

                shadow_indices = shadow_indices[np.unique(keep)]

        # Draw shadows first.
        for candidate_idx in shadow_indices:
            _draw_single_hand(
                ax=ax,
                hand=hands_frame[candidate_idx],
                hand_edges=hand_edges,
                color=style["shadow_color"],
                alpha=style["shadow_alpha"],
                linewidth=style["shadow_linewidth"],
                joint_size=style["shadow_joint_size"],
                zorder=1,
                style=style,
            )

        # Draw the closest sample last, so it remains visible.
        _draw_single_hand(
            ax=ax,
            hand=hands_frame[best_idx],
            hand_edges=hand_edges,
            color=style["best_color"],
            alpha=1.0,
            linewidth=style["best_linewidth"],
            joint_size=style["best_joint_size"],
            zorder=6,
            style=style,
        )

    _set_equal_axes(ax, axis_box=axis_box)
    ax.view_init(elev=elev, azim=azim)
    _style_hand_axis(ax, style=style)
    if title=="STAR-hand":
        title="CARD"
    ax.set_title(
        title,
        pad=2.5,
        color=style["label_color"],
        fontsize=9,
        fontweight="semibold",
    )


def _align_bundle_sample_to_reference(
    payload,
    sample_idx,
    reference_target,
    model_name,
    dataset_name,
    reference_dataset_name="assembly",
):
    """
    Validate compatibility after converting both targets to their plotting
    representations.

    HumanMAC and DLow-CVAE may store 20 joints without the root. Their plotting
    representation reconstructs the root as a zero-valued joint, producing the
    standard 21-joint representation.
    """
    if sample_idx >= len(payload["target"]):
        raise IndexError(
            f"sample_idx={sample_idx} is outside the saved samples for "
            f"{_display_model_name(model_name)}."
        )

    model_target_raw = np.asarray(payload["target"][sample_idx])
    reference_target_raw = np.asarray(reference_target)

    model_target, _ = _prepare_hands_for_plot(
        model_target_raw,
        model_name=model_name,
        dataset_name=dataset_name,
    )

    reference_target_prepared, _ = _prepare_hands_for_plot(
        reference_target_raw,
        model_name="ground_truth",
        dataset_name=reference_dataset_name,
    )

    if model_target.shape != reference_target_prepared.shape:
        raise ValueError(
            f"Target shape mismatch for {_display_model_name(model_name)} "
            f"after graph normalization: {model_target.shape} versus "
            f"reference {reference_target_prepared.shape}. "
            f"Raw shapes were {model_target_raw.shape} and "
            f"{reference_target_raw.shape}."
        )

    if not np.allclose(
        model_target,
        reference_target_prepared,
        rtol=1e-4,
        atol=1e-5,
    ):
        max_difference = np.max(
            np.abs(model_target - reference_target_prepared)
        )

        print(
            f"Warning: the normalized target stored for "
            f"{_display_model_name(model_name)} does not exactly match "
            f"the reference target. Maximum absolute difference: "
            f"{max_difference:.6g}. Check whether all evaluation bundles "
            f"use the same sample ordering and coordinate normalization."
        )

def create_saved_eval_video(
    sample_idx=0,
    dataset=None,
    action_filter=None,
    model_order=None,
    model_filter="all",
    output_path="hand_forecasting_comparison.mp4",
    fps=15,
    frame_stride=1,
    max_shadow_hands=32,
    num_columns=None,
    panel_width=3.0,
    panel_height=3.0,
    axis_mode="shared",
    axis_padding=0.08,
    elev=20,
    azim=-65,
    dpi=180,
    bitrate=3000,
    show_preview=False,
):
    """
    Create a video with one animated cell for GT and one for each competitor.

    For stochastic competitors:
      - the candidate with the lowest full-sequence MPJPE to GT is blue;
      - all other candidates are translucent grey shadows.

    Parameters
    ----------
    sample_idx : int
        Saved evaluation sample to visualize.

    output_path : str or Path
        Use .mp4 for H.264 video or .gif for an animated GIF.

    frame_stride : int
        Use every nth future frame. A value of 1 preserves all frames.

    axis_mode : {"shared", "per_panel"}
        "shared":
            all panels use the same coordinate limits, which makes motion
            magnitudes directly comparable.
        "per_panel":
            each panel has fixed limits computed from its own full sequence.

    num_columns : int or None
        Number of grid columns. When None, a near-square grid is used.
    """
    bundles = collect_saved_eval_samples(
        dataset=dataset,
        action_filter=action_filter,
    )
    bundles_by_model = _select_latest_bundle_per_model(bundles)

    if not bundles_by_model:
        raise ValueError(
            "No saved eval samples were found for the requested filters."
        )

    if model_order is None:
        candidate_model_names = list(bundles_by_model.keys())
    else:
        candidate_model_names = [
            name for name in model_order
            if name in bundles_by_model
        ]

    model_names = [
        name
        for name in candidate_model_names
        if _matches_model_filter(name, model_filter)
    ]

    if not model_names:
        available_models = ", ".join(
            _display_model_name(name)
            for name in candidate_model_names
        )
        raise ValueError(
            f"No models matched model_filter={model_filter!r}. "
            f"Available models: {available_models}"
        )

    first_bundle = bundles_by_model[model_names[0]]
    first_payload = first_bundle["payload"]

    if sample_idx >= len(first_payload["target"]):
        raise IndexError(
            f"sample_idx={sample_idx} is outside the "
            f"{len(first_payload['target'])} saved samples."
        )

    reference_dataset = first_bundle["metadata"].get(
        "dataset",
        dataset or "assembly",
    )
    reference_target = np.asarray(first_payload["target"][sample_idx])

    frame_stride = max(1, int(frame_stride))
    frame_indices = np.arange(
        0,
        reference_target.shape[0],
        frame_stride,
        dtype=int,
    )

    if frame_indices[-1] != reference_target.shape[0] - 1:
        frame_indices = np.append(
            frame_indices,
            reference_target.shape[0] - 1,
        )

    gt_sequence, gt_edges = _prepare_hands_for_plot(
        reference_target,
        model_name="ground_truth",
        dataset_name=reference_dataset,
        hand_edges=_hand_edges_for_plot(
            "ground_truth",
            reference_dataset,
        ),
    )

    # _prepare_hands_for_plot adds a hand/sample dimension for 2-D inputs,
    # but leaves [T, J, 3] sequences unchanged.
    if gt_sequence.ndim != 3:
        raise ValueError(
            f"Unexpected GT sequence shape: {gt_sequence.shape}"
        )

    panel_data = [
        {
            "name": "Ground Truth",
            "sequence": gt_sequence,
            "edges": gt_edges,
            "best_idx": None,
            "ground_truth": True,
        }
    ]

    for model_name in model_names:
        bundle = bundles_by_model[model_name]
        payload = bundle["payload"]
        model_dataset = bundle["metadata"].get(
            "dataset",
            reference_dataset,
        )

        _align_bundle_sample_to_reference(
            payload=payload,
            sample_idx=sample_idx,
            reference_target=reference_target,
            model_name=model_name,
            dataset_name=model_dataset,
            reference_dataset_name=reference_dataset,
        )

        model_edges = _hand_edges_for_plot(
            model_name,
            model_dataset,
        )

        pred_all = payload.get("pred_all")

        if pred_all is not None:
            sequence = np.asarray(pred_all[sample_idx])
            sequence, model_edges = _prepare_hands_for_plot(
                sequence,
                model_name=model_name,
                dataset_name=model_dataset,
                hand_edges=model_edges,
            )

            best_idx = _find_best_candidate_index(
                pred_all_sample=sequence,
                target_sample=payload["target"][sample_idx],
                model_name=model_name,
                dataset_name=model_dataset,
            )
        else:
            sequence = np.asarray(payload["pred"][sample_idx])
            sequence, model_edges = _prepare_hands_for_plot(
                sequence,
                model_name=model_name,
                dataset_name=model_dataset,
                hand_edges=model_edges,
            )

            if sequence.ndim == 3:
                sequence = sequence[None, ...]

            best_idx = 0

        if sequence.ndim != 4:
            raise ValueError(
                f"Unexpected prediction shape for "
                f"{_display_model_name(model_name)}: {sequence.shape}. "
                "Expected [K, T, J, 3]."
            )

        if sequence.shape[1] != reference_target.shape[0]:
            raise ValueError(
                f"Forecast-length mismatch for "
                f"{_display_model_name(model_name)}: "
                f"{sequence.shape[1]} versus {reference_target.shape[0]}."
            )

        panel_data.append(
            {
                "name": _display_model_name(model_name),
                "sequence": sequence,
                "edges": model_edges,
                "best_idx": best_idx,
                "ground_truth": False,
            }
        )

        print(
            f"{_display_model_name(model_name)}: "
            f"closest candidate index = {best_idx}"
        )

    # Fixed axis limits prevent visible zooming or camera jitter.
    if axis_mode == "shared":
        shared_box = _compute_sequence_axis_box(
            [panel["sequence"] for panel in panel_data],
            padding=axis_padding,
        )
        axis_boxes = [shared_box] * len(panel_data)

    elif axis_mode == "per_panel":
        axis_boxes = [
            _compute_sequence_axis_box(
                [panel["sequence"]],
                padding=axis_padding,
            )
            for panel in panel_data
        ]

    else:
        raise ValueError(
            "axis_mode must be either 'shared' or 'per_panel'."
        )

    num_competitors = len(panel_data) - 1

    # num_columns includes the dedicated Ground Truth column.
    if num_columns is None:
        competitor_columns = math.ceil(math.sqrt(num_competitors))
        num_columns = 1 + competitor_columns
    else:
        num_columns = max(2, int(num_columns))
        competitor_columns = num_columns - 1

    num_rows = max(
        1,
        math.ceil(num_competitors / competitor_columns),
    )

    fig = plt.figure(
        figsize=(
            panel_width * num_columns,
            panel_height * num_rows,
        ),
        facecolor="white",
    )

    grid = fig.add_gridspec(
        nrows=num_rows,
        ncols=num_columns,
        width_ratios=[1.0] + [1.0] * competitor_columns,
        wspace=0.0,
        hspace=0.02,
    )

    axes = []

    # Ground Truth is the only panel in the first column and spans all rows.
    gt_ax = fig.add_subplot(
        grid[:, 0],
        projection="3d",
    )
    axes.append(gt_ax)

    # Competitors are placed row by row in the remaining columns.
    for competitor_idx in range(num_competitors):
        row_idx = competitor_idx // competitor_columns
        column_idx = 1 + competitor_idx % competitor_columns

        competitor_ax = fig.add_subplot(
            grid[row_idx, column_idx],
            projection="3d",
        )
        axes.append(competitor_ax)

    # Hide unused cells in the competitor area.
    for empty_idx in range(num_competitors, num_rows * competitor_columns):
        row_idx = empty_idx // competitor_columns
        column_idx = 1 + empty_idx % competitor_columns

        empty_ax = fig.add_subplot(grid[row_idx, column_idx])
        empty_ax.axis("off")

    def update(animation_idx):
        source_frame_idx = int(frame_indices[animation_idx])

        for panel_idx, (ax, panel) in enumerate(
            zip(axes, panel_data)
        ):
            sequence = panel["sequence"]

            if panel["ground_truth"]:
                hands_frame = sequence[source_frame_idx]
            else:
                hands_frame = sequence[:, source_frame_idx]

            _draw_video_frame(
                ax=ax,
                hands_frame=hands_frame,
                hand_edges=panel["edges"],
                axis_box=axis_boxes[panel_idx],
                title=panel["name"],
                best_idx=panel["best_idx"],
                ground_truth=panel["ground_truth"],
                max_shadow_hands=max_shadow_hands,
                elev=elev,
                azim=azim,
                style=VIDEO_STYLE,
            )

        return []

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_indices),
        interval=1000 / float(fps),
        blit=False,
        repeat=True,
    )

    fig.subplots_adjust(
        left=0.015,
        right=0.985,
        top=0.93,
        bottom=0.055,
        wspace=0.00,
        hspace=0.02,
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    suffix = output_path.suffix.lower()

    if suffix == ".gif":
        writer = PillowWriter(fps=fps)
    elif suffix == ".mp4":
        if not plt.matplotlib.animation.writers.is_available("ffmpeg"):
            plt.close(fig)
            raise RuntimeError(
                "FFmpeg is not available. Install it or use an output "
                "filename ending in '.gif'."
            )

        writer = FFMpegWriter(
            fps=fps,
            codec="libx264",
            bitrate=bitrate,
            extra_args=[
                "-pix_fmt",
                "yuv420p",
                "-movflags",
                "+faststart",
            ],
        )
    else:
        plt.close(fig)
        raise ValueError(
            "output_path must end in '.mp4' or '.gif'."
        )

    animation.save(
        output_path,
        writer=writer,
        dpi=dpi,
    )

    print(f"Saved video to: {output_path.resolve()}")

    if show_preview:
        plt.show()
    else:
        plt.close(fig)

    return output_path


saved_eval_files = list_saved_eval_sample_files()
print(
    f"Found {len(saved_eval_files)} eval sample bundles "
    f"under {EVAL_SAMPLES_ROOT}"
)


video_path = create_saved_eval_video(
    sample_idx=0,
    dataset="assembly",
    action_filter="pick_up_screwd",
    output_path="outputs/pick_up_screwd_comparison.mp4",
    fps=15,
    frame_stride=1,
    max_shadow_hands=32,
    num_columns=3,
    axis_mode="shared",
    show_preview=False,
)

video_path

Found 68 eval sample bundles under /home/fagnelli/diffusion_hands/out/diffusion_hands_runs
Comusion: closest candidate index = 6
Gsps: closest candidate index = 9
Humanmac: closest candidate index = 0


In [25]:
from IPython.display import Video

Video(
    str(video_path),
    embed=True,
    html_attributes="controls loop autoplay muted",
)